# Analyse de Données Industrielles

Ce projet simule l'analyse de données de production d'une usine. L'objectif est de nettoyer les données, de les interroger en SQL pour en tirer des insights, de créer des visualisations pour comprendre les performances de l'entreprise, et enfin d'exporter un flux propre pour la construction d'un Dashboard Power BI.

## 1. Importation des librairies et chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os

# Configuration visuelle de base
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Chargement des données brutes
df_raw = pd.read_csv('industrial_data_raw.csv')

# Aperçu des données
display(df_raw.head())
display(df_raw.info())

## 2. Nettoyage des données (Data Cleaning)

Dans cette section, nous identifions et traitons les valeurs manquantes, les doublons et les valeurs aberrantes (outliers).

In [ ]:
df_clean = df_raw.copy()

# 1. Conversion des dates
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

# 2. Suppression des doublons
print(f"Doublons avant nettoyage : {df_clean.duplicated().sum()}")
df_clean = df_clean.drop_duplicates()

# 3. Traitement des valeurs manquantes
# Pour les capteurs (Température et Pression), on peut imputer par la médiane de la machine sur la ligne de production.
df_clean['Temperature_C'] = df_clean['Temperature_C'].fillna(df_clean.groupby('Machine_ID')['Temperature_C'].transform('median'))
df_clean['Pression_bar'] = df_clean['Pression_bar'].fillna(df_clean.groupby('Machine_ID')['Pression_bar'].transform('median'))

# Pour la quantité défectueuse, si elle manque on suppose 0 (optimiste) ou on drope la ligne.
df_clean['Quantite_Defectueuse'] = df_clean['Quantite_Defectueuse'].fillna(0)

# 4. Traitement des valeurs aberrantes
# Température ne peut pas être négative ou extrêmement élevée (>150)
valeurs_aberrantes_temp = df_clean[(df_clean['Temperature_C'] < 0) | (df_clean['Temperature_C'] > 150)]
print(f"Nb valeurs aberrantes Température : {len(valeurs_aberrantes_temp)}")
# Remplacement par NaN puis interpolation ou médiane : on remplace par la médiane de la machine
median_temp = df_clean.groupby('Machine_ID')['Temperature_C'].transform('median')
df_clean['Temperature_C'] = np.where((df_clean['Temperature_C'] < 0) | (df_clean['Temperature_C'] > 150), median_temp, df_clean['Temperature_C'])

print(f"\nTaille du dataset propre : {len(df_clean)} lignes")

## 3. Analyse avec SQL

L'utilisation de SQL est primordiale pour un Data Analyst. Nous allons insérer le dataframe dans une base de données locale `sqlite3` en mémoire pour l'interroger.

In [ ]:
# Connexion à une base de données en mémoire
conn = sqlite3.connect(':memory:')

# Insertion du dataframe dans la base
df_clean.to_sql('production', conn, index=False, if_exists='replace')

# Ajout d'une colonne Taux_Defaut dans la requête SQL
query = """
SELECT 
    Machine_ID,
    Ligne_Production,
    SUM(Heures_Arret) AS Total_Heures_Arret,
    SUM(Quantite_Defectueuse) * 100.0 / SUM(Quantite_Produite) AS Taux_Defaut_Moyen,
    AVG(Temperature_C) AS Temp_Moyenne
FROM 
    production
GROUP BY 
    Machine_ID, Ligne_Production
ORDER BY 
    Total_Heures_Arret DESC;
"""

df_sql_result = pd.read_sql_query(query, conn)
display(df_sql_result)

**Observation SQL :** 
Nous avons identifié les machines connaissant le plus de temps d'arrêt et analysé leur taux de défaut. Ces informations pourront être visualisées dans Power BI.

## 4. Visualisation des données (Python)

Visualisons la distribution des temps d'arrêt par machine et la corrélation éventuelle entre la température et les défauts.

In [ ]:
# 1. Temps d'arrêt par machine
plt.figure(figsize=(10,6))
sns.barplot(data=df_sql_result, x='Machine_ID', y='Total_Heures_Arret', palette='viridis')
plt.title("Volume total des temps d'arrêt par Machine")
plt.ylabel("Heures d'arrêt")
plt.xlabel("Machine")
plt.show()

# 2. Calcul du taux de défaut pour la corrélation
df_clean['Taux_Defaut'] = (df_clean['Quantite_Defectueuse'] / df_clean['Quantite_Produite']) * 100
df_clean['Taux_Defaut'] = df_clean['Taux_Defaut'].fillna(0)

# 3. Corrélation Température vs Taux de défaut
plt.figure(figsize=(10,6))
sns.scatterplot(data=df_clean, x='Temperature_C', y='Taux_Defaut', hue='Ligne_Production', alpha=0.5)
plt.title("Relation entre Température et Taux de Défaut")
plt.xlabel("Température (°C)")
plt.ylabel("Taux de Défaut (%)")
plt.show()

## 5. Export des données pour Dashboard Power BI

Afin de fournir la restitution visuelle demandée pour la Direction, nous allons exporter ce jeu de données propre sous format CSV. Ce fichier sera notre source d'importation dans Power BI.

In [ ]:
# Préparation des colonnes finales de manière optimisée
cols_to_export = [
    'Date', 'Machine_ID', 'Ligne_Production', 
    'Temperature_C', 'Pression_bar', 
    'Quantite_Produite', 'Quantite_Defectueuse', 'Taux_Defaut',
    'Heures_Fonctionnement', 'Heures_Arret'
]

df_export = df_clean[cols_to_export]

# Export
export_path = 'industrial_data_cleaned.csv'
df_export.to_csv(export_path, index=False, decimal='.', sep=',')

print(f"Les données propres ont été exportées vers : {export_path}")